> **Long run.** Same notebook with numbers of replicas, toy experiments and chain lengths close to those of the report.

# 04 · Frequentist analysis (4/4): local asymmetry in the Dalitz plane

**What this notebook does.** Is the asymmetry the same everywhere in phase space? Counting asymmetry in a fixed
$8\times8$ grid of $(m^2_{\rm lo},m^2_{\rm hi})$, homogeneity test, robustness (grids, signal window, charm vetoes),
null tests (charge permutation, phase-space simulation), magnet polarities, $\phi(1020)$ band, the region of the
published analysis and the charm vetoes.

| § | Content | Report section |
|---|---|---|
| 9 | Maps, homogeneity, null tests, polarities, $\phi$ band, published region, vetoes | Local asymmetry in the Dalitz plane (add.: null tests, region) |

**Scale.** A guided tour of the method, not the production run: replicas, toy experiments and chains are fewer than in
the report, so numbers can differ slightly from it; next to every result the notebook prints the value of the report
(**report:**). The notebook is self-contained and saves its figures in `figures/` and its numbers in `results/`.
Parts beyond the techniques of the course are tagged **[EXTRA]**.

## 0 · Common setup (data and nominal fit, as in notebook 01)

## 0 · Setup

In [ ]:
import os, time, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
from scipy.stats import norm, chi2
from iminuit import Minuit, cost

import plot_style
plot_style.use()
C = plot_style.COLORS
T0 = time.time()

# Data: put the three .root files of CERN Open Data record 4900 in ./data (or edit the path)
DATA_DIR = "data" if os.path.isdir("data") else "../../opendata-project/Data"
np.random.seed(1)

# figures and numerical results are saved next to the notebook
os.makedirs("figures", exist_ok=True); os.makedirs("results", exist_ok=True)
def save_fig(name):
    """Save the current figure in figures/ and show it."""
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

M_K = 493.677            # charged-kaon mass [MeV]
LO, HI = 5100.0, 5600.0  # nominal fit window of the mass [MeV]

## 1 · Data and preselection

The record contains $B^\pm\to h^\pm h^+h^-$ candidates (2011 data, 7 TeV), one file per magnet polarity. Every
candidate has the momentum, the particle-identification (PID) likelihoods and the charge of the three tracks
$H_1,H_2,H_3$. We read only the branches we need.

In [ ]:
tracks = (1, 2, 3)                     # the three final-state hadrons H1, H2, H3
# branches to read for each track: momentum, PID likelihoods, muon flag, charge
branches = [f"H{i}_{v}" for i in tracks for v in ("PX", "PY", "PZ", "ProbK", "ProbPi", "isMuon", "Charge")]

frames = []
for pol in ("Down", "Up"):             # one file per magnet polarity
    with uproot.open(f"{DATA_DIR}/B2HHH_Magnet{pol}.root") as f:
        d = f["DecayTree"].arrays(branches, library="pd")      # read the tree into a pandas DataFrame
    d["magnet"] = pol.lower()          # remember the polarity of each candidate
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
print("candidates in the record:", len(df))

**Preselection.** We want $B^\pm\to K^\pm K^+K^-$: all three tracks must look like kaons and not like pions or muons,

$$\texttt{ProbPi}<0.5\ \wedge\ \texttt{ProbK}>0.5\ \wedge\ \lnot\texttt{isMuon}\quad\text{for each of the three tracks.}$$

The cut flow below gives the effect of each requirement alone and cumulatively (additional result of the report).

In [ ]:
# boolean masks, True when the requirement is satisfied by ALL three tracks
not_muon = np.ones(len(df), bool)
pi_ok = np.ones(len(df), bool)
k_ok = np.ones(len(df), bool)
for i in tracks:
    not_muon &= ~df[f"H{i}_isMuon"].to_numpy().astype(bool)    # not identified as a muon
    pi_ok    &= df[f"H{i}_ProbPi"].to_numpy() < 0.5            # not pion-like
    k_ok     &= df[f"H{i}_ProbK"].to_numpy() > 0.5             # kaon-like

# cut flow: how many candidates pass each requirement alone, and applying them one after the other
print(f"{'requirement':28s} {'alone':>10s} {'cumulative':>11s}")
cum = np.ones(len(df), bool)
cutflow_rows = []
for name, mask in (("not muon (3 tracks)", not_muon), ("ProbPi < 0.5 (3 tracks)", pi_ok), ("ProbK > 0.5 (3 tracks)", k_ok)):
    cum &= mask
    print(f"{name:28s} {mask.sum():10d} {cum.sum():11d}")
    cutflow_rows.append((name, int(mask.sum()), int(cum.sum())))
print("report: 6311517 / 975724 / 28812 alone, 23895 cumulative")
pd.DataFrame(cutflow_rows, columns=["requirement", "alone", "cumulative"]).to_csv("results/01_cutflow.csv", index=False)

**Invariant mass.** We give the kaon mass to the three tracks, sum the four-momenta and take $m^2=E^2-|\vec p|^2$.
The two opposite-sign pairs $m(H_1H_2)$ and $m(H_1H_3)$ are kept for the Dalitz plane ($H_1$ is always the kaon with
charge opposite to the $B$).

We keep a slightly looser sample (`ProbK > 0.3`, `ProbPi < 0.7`) and store the *weakest* PID value of each
candidate: later the systematic study and the PID scan simply re-apply tighter cuts on these columns.

In [ ]:
loose = np.ones(len(df), bool)
for i in tracks:
    loose &= (df[f"H{i}_ProbK"] > 0.3) & (df[f"H{i}_ProbPi"] < 0.7) & ~df[f"H{i}_isMuon"].astype(bool)
d = df[loose].reset_index(drop=True)
del df

def invariant_mass(d, particles):
    """Invariant mass [MeV] of the listed tracks, kaon mass hypothesis."""
    E = sum(np.sqrt(d[f"H{i}_PX"]**2 + d[f"H{i}_PY"]**2 + d[f"H{i}_PZ"]**2 + M_K**2) for i in particles)
    px, py, pz = (sum(d[f"H{i}_{c}"] for i in particles) for c in ("PX", "PY", "PZ"))
    return np.sqrt(np.clip(E**2 - px**2 - py**2 - pz**2, 0, None))

d["m"]  = invariant_mass(d, (1, 2, 3))
d["m12"], d["m13"] = invariant_mass(d, (1, 2)), invariant_mass(d, (1, 3))
d["q"]  = d.H1_Charge + d.H2_Charge + d.H3_Charge              # charge of the B
d["pT"] = np.sqrt(sum(d[f"H{i}_PX"] for i in tracks)**2 + sum(d[f"H{i}_PY"] for i in tracks)**2)
d["p"]  = np.sqrt(d.pT**2 + sum(d[f"H{i}_PZ"] for i in tracks)**2)
d["min_probk"]  = d[[f"H{i}_ProbK" for i in tracks]].min(axis=1)
d["max_probpi"] = d[[f"H{i}_ProbPi" for i in tracks]].max(axis=1)
d = d[np.abs(d.q) == 1]
assert (d.H1_Charge == -d.q).all()          # H1 has the opposite charge of the B

nominal = d[(d.min_probk > 0.5) & (d.max_probpi < 0.5)]
sample = nominal[(nominal.m > LO) & (nominal.m < HI)].reset_index(drop=True)
print(f"preselected: {len(nominal)} (report 23895);  in the window [{LO:.0f},{HI:.0f}]: {len(sample)} (report 19731)")
m_pos = sample.m[sample.q > 0].to_numpy()       # B+
m_neg = sample.m[sample.q < 0].to_numpy()       # B-
n_pos, n_neg = len(m_pos), len(m_neg)

## 2 · Mass model and likelihood

For each charge the mass spectrum is a mixture of signal and background on the window $[5100,5600]$ MeV,

$$f^\pm(m)=N_s^\pm\,S(m;\mu,\sigma)+N_b^\pm\,B^\pm(m),\qquad
N_s^\pm=\tfrac{N_s}{2}(1\mp A),$$

* **signal** $S$: Gaussian, truncated and normalised to the window; $\mu,\sigma$ are *shared* by $B^+$ and $B^-$;
* **background** $B^\pm$: sum of two exponentials, $f\,E(\lambda_1)+(1-f)\,E(\lambda_2)$, with a shape and a yield of
  its own for each charge (the steep exponential describes the rise of the background towards the lower edge);
* **$A$** is a parameter of the fit: it is the *parameter of interest* (POI); the other 11 are *nuisance parameters*.

The likelihood is *extended* (the yields are parameters, the number of candidates is Poisson):

$$-\ln\mathcal L=(N_s+N_b)-\sum_{\pm}\sum_{i\in\pm}\ln\!\big[N_s^\pm S(m_i)+N_b^\pm B^\pm(m_i)\big]+\text{const.}$$

We write the pdfs ourselves (a few lines each) and let `iminuit` build the extended likelihood with
`cost.ExtendedUnbinnedNLL`: the model is a function `model(m, *parameters)` that returns the expected total number
of events and the density of events at $m$.

In [ ]:
def gauss_pdf(m, mu, sigma, lo=LO, hi=HI):
    """Gaussian normalised on [lo, hi]."""
    return norm.pdf(m, mu, sigma) / (norm.cdf(hi, mu, sigma) - norm.cdf(lo, mu, sigma))

def exp_pdf(m, lam, lo=LO, hi=HI):
    """Decreasing exponential exp(-lam (m - lo)) normalised on [lo, hi]."""
    return lam * np.exp(-lam * (m - lo)) / (1 - np.exp(-lam * (hi - lo)))

def bkg_pdf(m, lam1, lam2, f, lo=LO, hi=HI):
    """Background: two exponentials, weight f on the steep one (lam1 > lam2)."""
    return f * exp_pdf(m, lam1, lo, hi) + (1 - f) * exp_pdf(m, lam2, lo, hi)

# one model per charge; the parameters with the same name (N_s, A, mu, sigma) are shared in the joint fit
def model_pos(m, N_s, A, mu, sigma, N_b_pos, lam1_pos, lam2_pos, f_pos):
    n_sig = N_s * (1 - A) / 2                       # B+ : (1 - A) / 2
    return n_sig + N_b_pos, n_sig * gauss_pdf(m, mu, sigma) + N_b_pos * bkg_pdf(m, lam1_pos, lam2_pos, f_pos)

def model_neg(m, N_s, A, mu, sigma, N_b_neg, lam1_neg, lam2_neg, f_neg):
    n_sig = N_s * (1 + A) / 2                       # B- : (1 + A) / 2
    return n_sig + N_b_neg, n_sig * gauss_pdf(m, mu, sigma) + N_b_neg * bkg_pdf(m, lam1_neg, lam2_neg, f_neg)

### 3.2 Simultaneous fit: the nominal result

The two charges are fitted *together*: the total cost is the sum of the two extended likelihoods and $A$ is a
parameter of it, so the profile likelihood of $A$ is available directly. `Migrad` minimises, `Hesse` gives the
symmetric uncertainties from the curvature (inverse of the Fisher information) and `Minos` the asymmetric
intervals from the profile likelihood.

In [ ]:
def fit_joint(m_p, m_n, start, a_fixed=None):
    """Simultaneous fit of B+ and B-: the cost is the sum of the two extended likelihoods.
    With a_fixed the asymmetry is held at that value (profile likelihood at that point)."""
    nll = cost.ExtendedUnbinnedNLL(m_p, model_pos) + cost.ExtendedUnbinnedNLL(m_n, model_neg)
    f = Minuit(nll, **start)
    f.limits["A"] = (-1, 1); f.limits["mu"] = (5260, 5310); f.limits["sigma"] = (3, 40)
    f.limits["N_s"] = (0, 2 * (len(m_p) + len(m_n)))
    for s in ("pos", "neg"):
        f.limits[f"N_b_{s}"] = (0, 2 * (len(m_p) + len(m_n)))
        f.limits[f"lam1_{s}"] = (1e-5, 0.1); f.limits[f"lam2_{s}"] = (1e-5, 0.1); f.limits[f"f_{s}"] = (0, 1)
    if a_fixed is not None:
        f.values["A"] = a_fixed
        f.fixed["A"] = True
    f.migrad()
    return f

start = dict(N_s=0.78 * len(sample), A=-0.04, mu=5284.0, sigma=18.0,
             N_b_pos=0.22 * n_pos, lam1_pos=0.03, lam2_pos=0.002, f_pos=0.15,
             N_b_neg=0.22 * n_neg, lam1_neg=0.03, lam2_neg=0.002, f_neg=0.15)
fit = fit_joint(m_pos, m_neg, start)
best_vals_start = {k: fit.values[k] for k in start}      # best-fit values, used as starting point of later fits
fit.migrad(); fit.hesse()
fit.minos("A")                                      # 68 % interval (default cl = 1 sigma)
ci68 = (fit.values["A"] + fit.merrors["A"].lower, fit.values["A"] + fit.merrors["A"].upper)
fit_95 = copy.deepcopy(fit); fit_95.minos("A", cl=0.95)
ci95 = (fit_95.values["A"] + fit_95.merrors["A"].lower, fit_95.values["A"] + fit_95.merrors["A"].upper)

assert fit.valid and fit.values["lam1_pos"] > fit.values["lam2_pos"] and fit.values["lam1_neg"] > fit.values["lam2_neg"]
for k in ("N_s", "A", "mu", "sigma"):
    print(f"{k:8s} = {fit.values[k]:12.4f} +- {fit.errors[k]:.4f}")
print(f"N_b = {fit.values['N_b_pos'] + fit.values['N_b_neg']:.0f}")
print(f"\nA_raw = {fit.values['A']:+.4f} +- {fit.errors['A']:.4f} (Hesse)")
print(f"  68% Minos [{ci68[0]:+.4f}, {ci68[1]:+.4f}]    report: [-0.0526, -0.0356]")
print(f"  95% Minos [{ci95[0]:+.4f}, {ci95[1]:+.4f}]    report: [-0.0607, -0.0275]")
print("report: A = -0.0441 +- 0.0085, N_s = 15328 +- 133")

**Reading the result.** $A_{\rm raw}\approx-0.044$ with a Hesse uncertainty of $0.0085$: the asymmetry is about five standard deviations from zero. The Minos intervals are the ranges where the profile likelihood is within $1$ and $1.96^2$ units of its minimum; they are slightly asymmetric and the 95 % one does not contain zero.

In [ ]:
# table of the 12 parameters of the nominal fit, with their Hesse uncertainties, saved for later use
par_table = pd.DataFrame({"value": [fit.values[k] for k in fit.parameters], "hesse": [fit.errors[k] for k in fit.parameters]},
                         index=list(fit.parameters))
par_table.to_csv("results/01_fit_parameters.csv")
print(par_table.round(5).to_string())

### 3.3 Goodness of fit and picture

The $\chi^2=\sum_i(n_i-\nu_i)^2/\nu_i$ in bins of 10 MeV (both charges, $\nu_i$ from the fitted model) with
$\mathrm{ndof}=2\times50-12$. The pulls $(n_i-\nu_i)/\sqrt{\nu_i}$ show *where* the model fails. (The model is not
perfect: the signal tails are heavier than a Gaussian; $A_{\rm raw}$ does not depend on it, see §7.)

In [ ]:
def expected_counts(model, args, edges, n_sub=5):
    """Expected counts per bin: mean of the density at n_sub points in each bin, times the width."""
    width = edges[1] - edges[0]
    offs = (np.arange(n_sub) + 0.5) / n_sub                       # sub-points inside a bin
    pts = edges[:-1, None] + offs[None, :] * width                # shape (n_bins, n_sub)
    return model(pts, *args)[1].mean(axis=1) * width

edges = np.arange(LO, HI + 1e-9, 10.0)                            # bins of 10 MeV
centers = 0.5 * (edges[1:] + edges[:-1])
v = fit.values
# expected number of events in each bin for B+ and B-, from the fitted model
nu_pos = expected_counts(model_pos, [v[k] for k in ("N_s", "A", "mu", "sigma", "N_b_pos", "lam1_pos", "lam2_pos", "f_pos")], edges)
nu_neg = expected_counts(model_neg, [v[k] for k in ("N_s", "A", "mu", "sigma", "N_b_neg", "lam1_neg", "lam2_neg", "f_neg")], edges)
obs_pos, obs_neg = np.histogram(m_pos, edges)[0], np.histogram(m_neg, edges)[0]    # observed counts
chi2_val = np.sum((obs_pos - nu_pos)**2 / nu_pos) + np.sum((obs_neg - nu_neg)**2 / nu_neg)   # Pearson chi2
ndof = 2 * len(centers) - 12                                      # bins of both charges minus parameters
print(f"chi2/ndof = {chi2_val:.1f}/{ndof},  p = {chi2.sf(chi2_val, ndof):.1e}     (report: 160/88, p = 4.8e-6)")

# data with the fit (top) and pulls (bottom): the pulls show WHERE the model fails
pd.DataFrame({"m_center": centers, "obs_pos": obs_pos, "model_pos": nu_pos, "obs_neg": obs_neg, "model_neg": nu_neg}).to_csv(
    "results/01_mass_fit_bins.csv", index=False)    # data and model per bin
fig, ax = plt.subplots(2, 2, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for col, (obs, nu, lab, c) in enumerate(((obs_pos, nu_pos, "$B^+$", C["pos"]), (obs_neg, nu_neg, "$B^-$", C["neg"]))):
    ax[0, col].errorbar(centers, obs, np.sqrt(obs), fmt="o", ms=3, color="k", label="data")
    ax[0, col].plot(centers, nu, color=C["model"], label="fit")
    ax[0, col].set(yscale="log", ylabel="candidates / 10 MeV", title=lab); ax[0, col].legend()
    ax[1, col].bar(centers, (obs - nu) / np.sqrt(nu), width=9, color=c)
    ax[1, col].set(xlabel="m(KKK) [MeV]", ylabel="pull")
plt.tight_layout(); save_fig("04_joint_fit_pulls")

**Reading the result.** $\chi^2/{\rm ndof}$ is well above 1 and the $p$-value tiny: the model is **not** a perfect description of the spectrum (the pulls show structure on the flanks of the peak, where the signal tails are heavier than a Gaussian). What matters for the measurement is whether this affects $A$: the signal-model variants of notebook 03 say that it does not.

In [ ]:
# purity of the signal window |m - mu| < 2 sigma: signal / (signal + background) in the window, from the fit
mu_f, sg_f = fit.values["mu"], fit.values["sigma"]
x_w = np.linspace(mu_f - 2 * sg_f, mu_f + 2 * sg_f, 2001)
sig_in = fit.values["N_s"] * np.trapezoid(gauss_pdf(x_w, mu_f, sg_f), x_w)
bkg_in = sum(fit.values[f"N_b_{s}"] * np.trapezoid(bkg_pdf(x_w, fit.values[f"lam1_{s}"], fit.values[f"lam2_{s}"], fit.values[f"f_{s}"]), x_w)
             for s in ("pos", "neg"))
print(f"signal in the window: {sig_in:.0f}, background: {bkg_in:.0f}, purity {sig_in / (sig_in + bkg_in):.3f}   (report: about 96 %)")

In [ ]:
best_vals = dict(best_vals_start)      # best-fit values of the nominal fit, starting point of the later fits

In [ ]:
from iminuit.util import describe              # names of the arguments of a function

def make_models(lo=LO, hi=HI):
    """The nominal models of B+ and B- for the window [lo, hi]: the functions of section 2, for any window."""
    def model_pos(m, N_s, A, mu, sigma, N_b_pos, lam1_pos, lam2_pos, f_pos):
        n_sig = N_s * (1 - A) / 2
        return n_sig + N_b_pos, n_sig * gauss_pdf(m, mu, sigma, lo, hi) + N_b_pos * bkg_pdf(m, lam1_pos, lam2_pos, f_pos, lo, hi)
    def model_neg(m, N_s, A, mu, sigma, N_b_neg, lam1_neg, lam2_neg, f_neg):
        n_sig = N_s * (1 + A) / 2
        return n_sig + N_b_neg, n_sig * gauss_pdf(m, mu, sigma, lo, hi) + N_b_neg * bkg_pdf(m, lam1_neg, lam2_neg, f_neg, lo, hi)
    return model_pos, model_neg

def start_and_limits(name, n):
    """Starting value and limits of a parameter, decided by its name (n = number of candidates)."""
    if name == "N_s":              return 0.78 * n, (0, 2 * n)
    if name == "N_b":              return 0.22 * n, (0, 2 * n)               # background yield shared by the charges
    if name.startswith("N_b_"):    return 0.11 * n, (0, 2 * n)               # background yield of one charge
    if name == "A":                return -0.04, (-1, 1)
    if name.startswith("mu"):      return 5284.0, (5260, 5310)
    if name.startswith("sigma"):   return 18.0, (3, 40)
    if name == "beta":             return 1.8, (0.3, 5)                      # Crystal Ball tail
    if name == "m_cb":             return 10.0, (1.01, 50)
    if name.startswith("lam1"):    return 0.03, (1e-5, 0.1)                  # steep exponential
    if name.startswith("lam2"):    return 0.002, (1e-5, 0.1)                 # slow exponential
    if name.startswith("lam_"):    return 0.004, (1e-5, 0.1)                 # single exponential
    if name.startswith("f"):       return 0.15, (0, 1)
    raise ValueError(name)

def fit_two_charges(model_p, model_n, s):
    """Simultaneous fit of the two charges of the sample s with the given models (parameters with the same name are shared)."""
    mp, mn = s.m[s.q > 0].to_numpy(), s.m[s.q < 0].to_numpy()
    nll = cost.ExtendedUnbinnedNLL(mp, model_p) + cost.ExtendedUnbinnedNLL(mn, model_n)
    names_ = list(dict.fromkeys(describe(model_p)[1:] + describe(model_n)[1:]))        # parameters of the two models
    f = Minuit(nll, **{k: start_and_limits(k, len(s))[0] for k in names_})
    for k in names_:
        f.limits[k] = start_and_limits(k, len(s))[1]
    f.migrad(); f.hesse()
    return f

def fit_nominal(s, lo=LO, hi=HI):
    """Fit of the nominal model to the sample s in the window [lo, hi]."""
    return fit_two_charges(*make_models(lo, hi), s)

# check: this gives the nominal result of section 3
f_nom = fit_nominal(sample)
print(f"fit_nominal: A = {f_nom.values['A']:+.5f}   nominal fit: A = {fit.values['A']:+.5f}")
assert abs(f_nom.values["A"] - fit.values["A"]) < 1e-4

## 9 · Local asymmetry in the Dalitz plane

Is the asymmetry the same everywhere in phase space? Each candidate is placed in the **Dalitz plane** with the two
squared masses of the opposite-sign pairs, $m^2(H_1H_2)$ and $m^2(H_1H_3)$. Since $H_2,H_3$ are interchangeable
we sort them: $m^2_{\rm lo}=\min$, $m^2_{\rm hi}=\max$.

* Fixed grid $8\times8$ on $[0,16]\times[0,30]$ GeV$^2$, defined **before** looking at the asymmetry.
* In every bin with at least 20 candidates: $A_i=(N_i^--N_i^+)/N_i$, $\sigma(A_i)=\sqrt{(1-A_i^2)/N_i}$ and the
  significance with respect to the global asymmetry, $S_i=(A_i-A_{\rm glob})/\sigma(A_i)$.
* **Homogeneity test**: $\chi^2=\sum_iS_i^2$ with (number of bins $-1$) degrees of freedom.

This is a *counting* analysis: no background subtraction.

In [ ]:
# squared masses of the two opposite-sign K+K- pairs in GeV^2, sorted so that H2 and H3 are interchangeable
lo2 = (np.minimum(sample.m12, sample.m13) / 1e3)**2
hi2 = (np.maximum(sample.m12, sample.m13) / 1e3)**2
qv = sample.q.to_numpy()

def dalitz_map(lo2, hi2, q, min_events=20, nbins=8):
    """Counts and local asymmetry in the nbins x nbins grid; returns n, A_i, S_i, A_glob."""
    el, eh = np.linspace(0, 16, nbins + 1), np.linspace(0, 30, nbins + 1)       # fixed grid on [0,16] x [0,30] GeV^2
    n_pos = np.histogram2d(lo2[q > 0], hi2[q > 0], [el, eh])[0]                 # B+ candidates per bin
    n_neg = np.histogram2d(lo2[q < 0], hi2[q < 0], [el, eh])[0]                 # B- candidates per bin
    n = n_pos + n_neg
    a_glob = (np.sum(q < 0) - np.sum(q > 0)) / len(q)                           # global counting asymmetry
    with np.errstate(divide="ignore", invalid="ignore"):                        # empty bins give 0/0
        a_i = (n_neg - n_pos) / n                                               # local asymmetry
        s_i = (a_i - a_glob) / np.sqrt((1 - a_i**2) / n)                        # significance against the global one
    ok = n >= min_events                                                        # keep bins with at least 20 candidates
    return n, np.where(ok, a_i, np.nan), np.where(ok, s_i, np.nan), a_glob

n_i, a_i, s_i, a_glob = dalitz_map(lo2.to_numpy(), hi2.to_numpy(), qv)
ok = ~np.isnan(s_i)
chi2_h, ndof_h = np.nansum(s_i**2), ok.sum() - 1          # homogeneity statistic: sum of S_i^2, (bins - 1) d.o.f.
print(f"global A = {a_glob:+.4f};  populated bins: {ok.sum()};  bins with |S|>3: {np.sum(np.abs(s_i[ok]) > 3)}")
print(f"homogeneity: chi2 = {chi2_h:.1f} for {ndof_h} d.o.f., p = {chi2.sf(chi2_h, ndof_h):.1e}     (report: 80.8/31, p = 3e-6)")

# three maps: candidates per bin, local asymmetry A_i, significance S_i
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for a, (img, lab, kw) in zip(ax, ((np.where(n_i > 0, n_i, np.nan), "candidates per bin", dict(cmap="Greys")),
                                  (a_i, "$A_i$", dict(cmap="coolwarm", vmin=-0.5, vmax=0.5)),
                                  (s_i, "$S_i$", dict(cmap="coolwarm", vmin=-4, vmax=4)))):
    im = a.imshow(img.T, origin="lower", aspect="auto", extent=[0, 16, 0, 30], **kw)
    plt.colorbar(im, ax=a, label=lab); a.set(xlabel="$m^2_{lo}$ [GeV$^2$]", ylabel="$m^2_{hi}$ [GeV$^2$]"); a.grid(False)
plt.tight_layout(); save_fig("04_dalitz_maps")

**Null test by charge permutation** [EXTRA]. If we shuffle the charge labels among the candidates, the Dalitz
distribution and the global asymmetry are unchanged but any *local* asymmetry is destroyed: $\chi^2$ must then follow
its distribution under uniformity. The observed value must lie far in the tail. (Report: 2000 permutations.)

In [ ]:
chi2_perm = []
for _ in range(2000):
    q_shuffled = np.random.permutation(qv)                # shuffle the charge labels among the candidates
    _, _, s_p, _ = dalitz_map(lo2.to_numpy(), hi2.to_numpy(), q_shuffled)
    chi2_perm.append(np.nansum(s_p**2))                   # homogeneity statistic of the shuffled sample
chi2_perm = np.array(chi2_perm)
print(f"permutations: mean {chi2_perm.mean():.1f} +- {chi2_perm.std():.1f}, max {chi2_perm.max():.1f};  "
      f"observed {chi2_h:.1f}, fraction >= observed: {np.mean(chi2_perm >= chi2_h):.4f}   (report: 31.5 +- 8.4, none above 80.8)")

**Reading the result.** Shuffling the charges gives a $\chi^2$ distribution centred near its degrees of freedom and never reaching the observed value: the local asymmetry is **not uniform** over the Dalitz plane.

In [ ]:
# permutation distribution against the chi2 law of a uniform asymmetry, with the observed value
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(chi2_perm, bins=30, density=True, color=C["freq"], alpha=0.6, label=f"{len(chi2_perm)} charge permutations")
x = np.linspace(0, max(chi2_h * 1.1, chi2_perm.max()), 300); ax.plot(x, chi2.pdf(x, ndof_h), "k--", label=f"$\\chi^2$, {ndof_h} d.o.f.")
ax.axvline(chi2_h, color=C["model"], label=f"observed: {chi2_h:.1f}")
ax.set(xlabel="homogeneity statistic $\\chi^2=\\sum_i S_i^2$", ylabel="density"); ax.legend()
plt.tight_layout(); save_fig("04_permutation_test")
np.save("results/04_chi2_permutations.npy", chi2_perm)

**Robustness of the homogeneity test.** The conclusion must not depend on the grid, nor be driven by the background
or by the charm resonances: we repeat the test with other grids, in the signal window alone ($|m-\mu|<2\sigma$, where
the background is small) and with the charm vetoes (defined at the end of this section).

**Null test with the phase-space simulation.** The simulation of $B^\pm\to K^\pm K^+K^-$ has *no asymmetry by
construction*: the same procedure on it must give a $\chi^2$ compatible with its degrees of freedom. We apply it to
subsamples of the size of the data and to the whole simulation.

In [ ]:
def homogeneity(lo2_, hi2_, q_, nbins=8):
    """Homogeneity statistic of a sample: sum of S_i^2, degrees of freedom and number of bins with |S| > 3."""
    n_, a_, s_, ag_ = dalitz_map(np.asarray(lo2_), np.asarray(hi2_), np.asarray(q_), nbins=nbins)
    k = ~np.isnan(s_)
    return np.nansum(s_**2), k.sum() - 1, np.sum(np.abs(s_[k]) > 3)

# charm resonances decaying to K+K-: (mass, half-width of the veto) in MeV
CHARM = {"D0": (1864.84, 30.0), "Jpsi": (3096.9, 30.0), "chic0": (3414.7, 60.0)}
def veto_mask(s, states):
    """True for the candidates kept: neither opposite-sign pair is within the window of a vetoed resonance."""
    keep = np.ones(len(s), bool)
    for st in states:
        m0, hw = CHARM[st]
        keep &= ~((np.abs(s.m12 - m0) < hw) | (np.abs(s.m13 - m0) < hw))
    return keep

def lo_hi(s):
    return (np.minimum(s.m12, s.m13) / 1e3)**2, (np.maximum(s.m12, s.m13) / 1e3)**2

# the same test on different grids and different samples
tests = {}
for nb in (6, 8, 10, 12):
    tests[f"{nb}x{nb} grid"] = homogeneity(lo2, hi2, qv, nb)
sw = (np.abs(sample.m - fit.values["mu"]) < 2 * fit.values["sigma"]).to_numpy()      # signal window: little background
tests["signal window"] = homogeneity(lo2[sw], hi2[sw], qv[sw])
for lab, states in (("D0 veto", ["D0"]), ("D0, J/psi, chic0 vetoes", ["D0", "Jpsi", "chic0"])):
    kv = veto_mask(sample, states)
    tests[lab] = homogeneity(lo2[kv], hi2[kv], qv[kv])
print(f"{'sample / grid':26s} {'chi2':>7s} {'ndof':>5s} {'p':>9s} {'bins |S|>3':>11s}")
for lab, (c2_, nd_, n3) in tests.items():
    print(f"{lab:26s} {c2_:7.1f} {nd_:5d} {chi2.sf(c2_, nd_):9.1e} {n3:11d}")
print("report: the homogeneity test holds in the signal window, with the vetoes and for other grids (all p << 1e-3)")

# phase-space simulation: no asymmetry by construction, so chi2 must be compatible with its d.o.f.
with uproot.open(f"{DATA_DIR}/PhaseSpaceSimulation.root") as f:
    sim = f["PhaseSpaceTree"].arrays([f"H{i}_{v}" for i in tracks for v in ("PX", "PY", "PZ", "Charge")], library="pd")
sim["m12"], sim["m13"] = invariant_mass(sim, (1, 2)), invariant_mass(sim, (1, 3))
sim["q"] = sim.H1_Charge + sim.H2_Charge + sim.H3_Charge
sim = sim[np.abs(sim.q) == 1].reset_index(drop=True)
lo_s, hi_s = lo_hi(sim)
c2_all, nd_all, _ = homogeneity(lo_s, hi_s, sim.q)                   # the whole simulation
c2_sub = []
for _ in range(500):                                                 # subsamples with the size of the data
    idx = np.random.choice(len(sim), len(sample), replace=False)
    c2_sub.append(homogeneity(lo_s.to_numpy()[idx], hi_s.to_numpy()[idx], sim.q.to_numpy()[idx])[0])
print(f"phase-space simulation ({len(sim)} decays): whole sample chi2 = {c2_all:.1f} for {nd_all} d.o.f. (p = {chi2.sf(c2_all, nd_all):.2f})")
print(f"  100 subsamples of {len(sample)}: chi2 = {np.mean(c2_sub):.1f} +- {np.std(c2_sub):.1f}  (expected {nd_all} +- {np.sqrt(2 * nd_all):.1f})"
      "   (report: consistent with no asymmetry, slightly optimistic p-values)")

In [ ]:
pd.DataFrame([dict(sample=k, chi2=v[0], ndof=v[1], n_bins_3sigma=v[2], p=chi2.sf(v[0], v[1])) for k, v in tests.items()]).to_csv(
    "results/04_homogeneity_tests.csv", index=False)
np.save("results/04_chi2_phase_space_subsamples.npy", np.array(c2_sub))

**Same structure in the two magnet polarities.** The maps of the two polarities are compared bin by bin: Pearson
correlation of $A_i$ and $\chi^2$ of the bin-by-bin difference $(A_i^{\rm down}-A_i^{\rm up})/\sqrt{\sigma_{\rm down}^2+\sigma_{\rm up}^2}$.

In [ ]:
maps = {}
for pol in ("down", "up"):
    mpol = (sample.magnet == pol).to_numpy()
    maps[pol] = dalitz_map(lo2.to_numpy()[mpol], hi2.to_numpy()[mpol], qv[mpol])    # one map per polarity
n_d, a_d, _, _ = maps["down"]; n_u, a_u, _, _ = maps["up"]
both = ~np.isnan(a_d) & ~np.isnan(a_u)                              # bins populated in both maps
# bin-by-bin difference in units of its uncertainty
z = (a_d[both] - a_u[both]) / np.sqrt((1 - a_d[both]**2) / n_d[both] + (1 - a_u[both]**2) / n_u[both])
print(f"bins populated in both: {both.sum()};  Pearson correlation of A_i: {np.corrcoef(a_d[both], a_u[both])[0, 1]:+.2f};  "
      f"chi2 of the difference = {np.sum(z**2):.1f} for {both.sum()} bins (p = {chi2.sf(np.sum(z**2), both.sum()):.2f})")
print("report: the two maps agree bin by bin (chi2 = 23.0 for 29 bins, p = 0.78)")

**Reading the result.** The two polarities show the same pattern bin by bin (positive correlation and a $\chi^2$ of the difference compatible with its degrees of freedom): the structure is not a detector asymmetry that flips with the polarity.

**Where is the asymmetry?** The $\phi(1020)$ appears as a band at the kinematic edge ($m^2_{\rm lo}\approx1.04$ GeV$^2$)
and the charm resonances ($D^0$, $J/\psi$, $\chi_{c0}$) as bands. We count the asymmetry in the $\phi$ band and in
the bins that are significant. These numbers describe *a posteriori* a region selected because it is significant:
an illustration, not a new significance.

In [ ]:
def count_region(mask):
    q_ = qv[mask]; a = (np.sum(q_ < 0) - np.sum(q_ > 0)) / len(q_)
    return len(q_), a, np.sqrt((1 - a**2) / len(q_))

m_lo_mev = np.minimum(sample.m12, sample.m13).to_numpy()
phi = np.abs(m_lo_mev - 1019.461) < 10                                   # phi(1020) band
bins_sig = (lo2 < 4) & (hi2 > 7.5) & (hi2 < 11.25)                        # the two bins with |S| > 3
bins_sig = bins_sig.to_numpy()
for lab, mask in (("all candidates", np.ones(len(qv), bool)), ("phi(1020) band", phi),
                  ("significant bins", bins_sig), ("significant bins without phi", bins_sig & ~phi)):
    n_, a_, da_ = count_region(mask)
    print(f"{lab:30s} n = {n_:6d}   A_count = {a_:+.4f} +- {da_:.4f}")
print("report: phi band -0.004 +- 0.023;  significant bins without phi -0.145 +- 0.019")

In [ ]:
# counting asymmetry in the regions described a posteriori, saved and drawn; and the phi(1020) band
regions = {"all candidates": np.ones(len(qv), bool), "phi(1020) band": phi, "significant bins": bins_sig,
           "significant bins without phi": bins_sig & ~phi}
reg_rows = []
for lab, mask in regions.items():
    n_, a_, da_ = count_region(mask)
    reg_rows.append(dict(region=lab, n=n_, A_count=a_, err=da_))
reg_tab = pd.DataFrame(reg_rows)
reg_tab.to_csv("results/04_regions_counting.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.2]})
ax[0].hist(m_lo_mev, bins=np.arange(980, 1100.1, 5), histtype="step", color="k")
ax[0].axvspan(1019.461 - 10, 1019.461 + 10, color="0.85", label="$\\phi(1020)$ band")
ax[0].set(xlabel="$m_{lo}(K^+K^-)$ [MeV]", ylabel="candidates / 5 MeV"); ax[0].legend()
yy = np.arange(len(reg_tab))[::-1]
ax[1].errorbar(reg_tab.A_count, yy, reg_tab.err, fmt="o", color=C["freq"], capsize=3)
ax[1].set_yticks(yy); ax[1].set_yticklabels(reg_tab.region); ax[1].axvline(0, color="k", lw=0.8)
ax[1].set(xlabel="counting asymmetry")
plt.tight_layout(); save_fig("04_phi_band_and_regions")

**Magnet polarity.** The left–right detection asymmetry changes sign with the magnet polarity, while a CP asymmetry
does not. The counting asymmetry of the significant region in each polarity:

In [ ]:
for pol in ("down", "up"):
    mask = bins_sig & (sample.magnet == pol).to_numpy()      # the significant region, one polarity at a time
    n_, a_, da_ = count_region(mask)
    print(f"magnet {pol:4s}: n = {n_}, A_count = {a_:+.3f} +- {da_:.3f}")
print("report: -0.129 +- 0.023 (down), -0.126 +- 0.028 (up)")

### Region of the published analysis

The LHCb publication on the same data measured the asymmetry in the region $1.2<m^2_{\rm lo}<2.0$ GeV$^2$,
$m^2_{\rm hi}<15$ GeV$^2$ and found $A_{\rm raw}=-0.239\pm0.020$. The region is fixed by the paper, not chosen on
our data, so the comparison is *a priori*. We fit the nominal model to the candidates in the region.

In [ ]:
in_paper = ((lo2 > 1.2) & (lo2 < 2.0) & (hi2 < 15)).to_numpy()     # region of the published analysis (GeV^2)
reg = sample[in_paper]
f_reg = fit_nominal(reg)                                         # nominal model fitted to the candidates of the region
print(f"candidates in the region: {len(reg)}  (report: 2146)")
print(f"A_raw (region) = {f_reg.values['A']:+.3f} +- {f_reg.errors['A']:.3f}     (report, frequentist: -0.237 +- 0.025; "
      f"published: -0.239 +- 0.020)")

In [ ]:
# Dalitz plot with the region of the published analysis and the resonances; mass spectra of B+ and B- in the region
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].hist2d(lo2, hi2, bins=100, range=[[0, 16], [0, 30]], cmap="Greys", norm=plt.matplotlib.colors.LogNorm())
ax[0].add_patch(plt.Rectangle((1.2, 0), 0.8, 15, fill=False, ec=C["model"], lw=2))
for m0 in (1.0195**2, 1.86484**2, 3.0969**2, 3.4147**2):
    ax[0].axvline(m0, color="gray", ls="--", lw=0.7); ax[0].axhline(m0, color="gray", ls="--", lw=0.7)
ax[0].set(xlabel="$m^2_{lo}$ [GeV$^2$]", ylabel="$m^2_{hi}$ [GeV$^2$]"); ax[0].grid(False)
bins_r = np.arange(LO, HI + 1, 10)
ax[1].hist(reg.m[reg.q > 0], bins_r, histtype="step", color=C["pos"], label="$B^+$")
ax[1].hist(reg.m[reg.q < 0], bins_r, histtype="step", color=C["neg"], label="$B^-$")
ax[1].set(xlabel="m(KKK) [MeV]", ylabel="candidates / 10 MeV"); ax[1].legend()
plt.tight_layout(); save_fig("04_published_region")

### Charm vetoes

$B^\pm\to D^0K^\pm$, $J/\psi K^\pm$, $\chi_{c0}K^\pm$ with the resonance decaying to $K^+K^-$ are not charmless. A
candidate is rejected if one of its two opposite-sign pairs falls within $\pm30$ MeV ($D^0$, $J/\psi$) or $\pm60$ MeV
($\chi_{c0}$) of the mass of the resonance. The vetoes are a robustness check, not part of the systematic.

In [ ]:
# (CHARM and veto_mask are defined in the robustness cell above)
veto_rows = []
for lab, states in (("D0 veto", ["D0"]), ("D0, J/psi, chic0 vetoes", ["D0", "Jpsi", "chic0"])):
    s = sample[veto_mask(sample, states)]                  # candidates that survive the veto
    f = fit_nominal(s)                                    # nominal model on the vetoed sample
    veto_rows.append(dict(sample=lab, n=len(s), A=f.values["A"], err=f.errors["A"]))
    print(f"{lab:26s} n = {len(s):6d}   A_raw = {f.values['A']:+.4f} +- {f.errors['A']:.4f}")
pd.DataFrame(veto_rows + [dict(sample="published region", n=len(reg), A=f_reg.values["A"], err=f_reg.errors["A"])]).to_csv(
    "results/04_vetoes_and_region_fits.csv", index=False)
print("report: n = 18744, -0.0493 +- 0.0087  and  n = 16367, -0.0421 +- 0.0094")

## Results saved

In [ ]:
summary = dict(
    homogeneity=dict(chi2=float(chi2_h), ndof=int(ndof_h), p=float(chi2.sf(chi2_h, ndof_h))),
    permutation=dict(mean=float(chi2_perm.mean()), sd=float(chi2_perm.std()), max=float(chi2_perm.max())),
    tests={k: dict(chi2=float(v[0]), ndof=int(v[1]), n_3sigma=int(v[2])) for k, v in tests.items()},
    published_region=dict(n=int(len(reg)), A=float(f_reg.values["A"]), err=float(f_reg.errors["A"])))
json.dump(summary, open("results/04_summary.json", "w"), indent=1)
np.savez("results/04_dalitz_map.npz", n=n_i, A=a_i, S=s_i)
print("saved results/04_summary.json; figures in figures/")

## Summary
* The asymmetry is **not uniform** over the Dalitz plane: the homogeneity $\chi^2$ is far above its degrees of
  freedom, also in the signal window, with the vetoes and for other grids; charge permutations never reach it and the
  phase-space simulation (no asymmetry) is consistent with uniformity.
* The structure is the same in the two magnet polarities; it is **not produced by the $\phi(1020)$**.
* In the region fixed by the published analysis the fit gives $A_{\rm raw}\approx-0.24$, reproducing the published
  value. The asymmetries are counts without background subtraction and part of the variation can be instrumental.

In [ ]:
print(f"Running time of this notebook: {(time.time() - T0) / 60:.1f} min")